# Анализ датасета CADICA 50+ (стенозы ≥50%)

Датасет получен из CADICA путём ремаппинга классов:
- **Класс 0 (`stenosis_0`)**: стеноз 50–70%
- **Класс 1 (`stenosis_1`)**: стеноз >70% (объединение 70–90%, 90–98%, 99%, 100%)
- Стенозы <50% убраны (кадры остаются, но без аннотаций)

Формат имён файлов: `p{patient}_v{video}_{frame}.png`
- **Пациент**: `p{N}` (e.g. `p10`)
- **Последовательность (видео)**: `p{N}_v{M}` (e.g. `p10_v1`)
- **Кадр**: последние 5 цифр (e.g. `00016`)

## 1. Импорт библиотек и настройка путей

In [ ]:
import os
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")

DATASET_DIR = Path("/home/dsa/stenosis/data/cadica_split_50plus")
SPLITS = ["train", "valid", "test"]
CLASS_NAMES = {0: "stenosis_0 (50-70%)", 1: "stenosis_1 (>70%)"}

print(f"Dataset dir: {DATASET_DIR}")
print(f"Exists: {DATASET_DIR.exists()}")
for split in SPLITS:
    n_img = len(list((DATASET_DIR / split / "images").glob("*.png")))
    n_lbl = len(list((DATASET_DIR / split / "labels").glob("*.txt")))
    print(f"  {split}: {n_img} images, {n_lbl} labels")

## 2. Сканирование файлов и примеры имён

In [ ]:
all_images = []
for split in SPLITS:
    img_dir = DATASET_DIR / split / "images"
    for f in sorted(img_dir.glob("*.png")):
        all_images.append((split, f.name))

print(f"Всего изображений: {len(all_images)}")
print(f"\nПримеры имён файлов:")
for _, name in all_images[:5]:
    print(f"  {name}")
print("  ...")
for _, name in all_images[-3:]:
    print(f"  {name}")

## 3. Парсинг имён файлов и создание DataFrame

In [ ]:
records = []

for split in SPLITS:
    img_dir = DATASET_DIR / split / "images"
    lbl_dir = DATASET_DIR / split / "labels"
    
    for img_path in sorted(img_dir.glob("*.png")):
        name = img_path.stem  # e.g. "p10_v1_00016"
        # Parse: p{patient}_v{video}_{frame}
        m = re.match(r'(p\d+)_(v\d+)_(\d+)', name)
        patient_id = m.group(1)
        video_id = m.group(2)
        sequence_id = f"{patient_id}_{video_id}"
        frame = m.group(3)
        
        label_path = lbl_dir / f"{name}.txt"
        has_label = label_path.exists()
        has_stenosis = False
        num_bboxes = 0
        n_class0 = 0  # stenosis 50-70%
        n_class1 = 0  # stenosis >70%
        
        if has_label and label_path.stat().st_size > 0:
            content = label_path.read_text().strip()
            if content:
                has_stenosis = True
                for line in content.splitlines():
                    cls = int(line.split()[0])
                    num_bboxes += 1
                    if cls == 0:
                        n_class0 += 1
                    elif cls == 1:
                        n_class1 += 1
        
        records.append({
            'split': split,
            'image_name': img_path.name,
            'patient_id': patient_id,
            'sequence_id': sequence_id,
            'frame': frame,
            'has_stenosis': has_stenosis,
            'num_bboxes': num_bboxes,
            'n_class0': n_class0,
            'n_class1': n_class1,
        })

df = pd.DataFrame(records)
print(f"DataFrame shape: {df.shape}")
df.head(10)

## 4. Общая статистика: последовательности, кадры, пациенты

In [ ]:
n_patients = df['patient_id'].nunique()
n_sequences = df['sequence_id'].nunique()
n_frames = len(df)

print(f"{'='*55}")
print(f"  Общая статистика датасета CADICA 50+")
print(f"{'='*55}")
print(f"  Пациентов:            {n_patients}")
print(f"  Последовательностей:  {n_sequences}")
print(f"  Кадров (изображений): {n_frames}")
print(f"{'='*55}")

# По сплитам
for split in SPLITS:
    sdf = df[df['split'] == split]
    print(f"\n  [{split}] пациентов: {sdf['patient_id'].nunique()}, "
          f"последовательностей: {sdf['sequence_id'].nunique()}, "
          f"кадров: {len(sdf)}")

# Кадров на последовательность
frames_per_seq = df.groupby('sequence_id').size()
print(f"\nКадров на последовательность:")
print(f"  min:    {frames_per_seq.min()}")
print(f"  max:    {frames_per_seq.max()}")
print(f"  mean:   {frames_per_seq.mean():.1f}")
print(f"  median: {frames_per_seq.median():.1f}")

# Последовательностей на пациента
seqs_per_patient = df.groupby('patient_id')['sequence_id'].nunique()
print(f"\nПоследовательностей на пациента:")
print(f"  min:    {seqs_per_patient.min()}")
print(f"  max:    {seqs_per_patient.max()}")
print(f"  mean:   {seqs_per_patient.mean():.1f}")
print(f"  median: {seqs_per_patient.median():.1f}")

## 5. Классификация последовательностей по наличию стеноза

Последовательность считается **со стенозом**, если хотя бы один кадр имеет непустой label.
Также различаем классы: stenosis_0 (50–70%) и stenosis_1 (>70%).

In [ ]:
# Классификация кадров
n_frames_with_stenosis = df['has_stenosis'].sum()
n_frames_without = len(df) - n_frames_with_stenosis
print(f"Кадры со стенозом (≥50%): {n_frames_with_stenosis} ({n_frames_with_stenosis/len(df)*100:.1f}%)")
print(f"Кадры без стеноза:        {n_frames_without} ({n_frames_without/len(df)*100:.1f}%)")

# По классам
total_c0 = df['n_class0'].sum()
total_c1 = df['n_class1'].sum()
print(f"\nВсего bbox-ов класса 0 (50-70%): {total_c0}")
print(f"Всего bbox-ов класса 1 (>70%):   {total_c1}")

# Классификация последовательностей
seq_stats = df.groupby('sequence_id').agg(
    total_frames=('has_stenosis', 'size'),
    stenosis_frames=('has_stenosis', 'sum'),
    total_bboxes=('num_bboxes', 'sum'),
    bboxes_class0=('n_class0', 'sum'),
    bboxes_class1=('n_class1', 'sum'),
    patient=('patient_id', 'first'),
    split=('split', 'first'),
).reset_index()
seq_stats['stenosis_ratio'] = seq_stats['stenosis_frames'] / seq_stats['total_frames']
seq_stats['has_any_stenosis'] = seq_stats['stenosis_frames'] > 0
seq_stats['has_class0'] = seq_stats['bboxes_class0'] > 0
seq_stats['has_class1'] = seq_stats['bboxes_class1'] > 0

n_seq_with = seq_stats['has_any_stenosis'].sum()
n_seq_without = len(seq_stats) - n_seq_with
print(f"\nПоследовательности со стенозом:  {n_seq_with} ({n_seq_with/len(seq_stats)*100:.1f}%)")
print(f"Последовательности без стеноза:  {n_seq_without} ({n_seq_without/len(seq_stats)*100:.1f}%)")

n_seq_c0 = seq_stats['has_class0'].sum()
n_seq_c1 = seq_stats['has_class1'].sum()
print(f"\nПоследовательности с классом 0 (50-70%): {n_seq_c0}")
print(f"Последовательности с классом 1 (>70%):   {n_seq_c1}")

print(f"\nДоля кадров со стенозом (среди последовательностей со стенозом):")
with_st = seq_stats[seq_stats['has_any_stenosis']]
print(f"  min:    {with_st['stenosis_ratio'].min():.1%}")
print(f"  max:    {with_st['stenosis_ratio'].max():.1%}")
print(f"  mean:   {with_st['stenosis_ratio'].mean():.1%}")
print(f"  median: {with_st['stenosis_ratio'].median():.1%}")

seq_stats.head(15)

## 6. Анализ распределения по пациентам

In [ ]:
patient_stats = seq_stats.groupby('patient').agg(
    total_sequences=('sequence_id', 'count'),
    sequences_with_stenosis=('has_any_stenosis', 'sum'),
    total_frames=('total_frames', 'sum'),
    stenosis_frames=('stenosis_frames', 'sum'),
    total_bboxes=('total_bboxes', 'sum'),
    bboxes_class0=('bboxes_class0', 'sum'),
    bboxes_class1=('bboxes_class1', 'sum'),
).reset_index()
patient_stats['sequences_without_stenosis'] = patient_stats['total_sequences'] - patient_stats['sequences_with_stenosis']
patient_stats['patient_has_stenosis'] = patient_stats['sequences_with_stenosis'] > 0

n_patients_with = patient_stats['patient_has_stenosis'].sum()
n_patients_without = len(patient_stats) - n_patients_with

print(f"Пациенты со стенозом (≥50%): {n_patients_with} ({n_patients_with/len(patient_stats)*100:.1f}%)")
print(f"Пациенты без стеноза:        {n_patients_without} ({n_patients_without/len(patient_stats)*100:.1f}%)")
print(f"\nСводная таблица по пациентам:")
print(patient_stats.to_string(index=False))

## 7. Визуализация статистики датасета

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Pie chart: последовательности со стенозом vs без
ax = axes[0, 0]
labels_pie = ['Со стенозом', 'Без стеноза']
sizes = [n_seq_with, n_seq_without]
colors_pie = ['#e74c3c', '#2ecc71']
ax.pie(sizes, labels=labels_pie, autopct='%1.1f%%', colors=colors_pie, startangle=90, textprops={'fontsize': 12})
ax.set_title(f'Последовательности: стеноз vs норма\n(всего {len(seq_stats)})', fontsize=13)

# 2. Pie chart: кадры со стенозом vs без
ax = axes[0, 1]
sizes2 = [n_frames_with_stenosis, n_frames_without]
ax.pie(sizes2, labels=labels_pie, autopct='%1.1f%%', colors=colors_pie, startangle=90, textprops={'fontsize': 12})
ax.set_title(f'Кадры: стеноз vs норма\n(всего {n_frames})', fontsize=13)

# 3. Гистограмма: кадров на последовательность
ax = axes[1, 0]
ax.hist(frames_per_seq.values, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
ax.set_xlabel('Кадров в последовательности')
ax.set_ylabel('Количество последовательностей')
ax.set_title('Распределение кадров на последовательность')
ax.axvline(frames_per_seq.mean(), color='red', linestyle='--', label=f'mean={frames_per_seq.mean():.0f}')
ax.legend()

# 4. Гистограмма: последовательностей на пациента
ax = axes[1, 1]
ax.hist(seqs_per_patient.values, bins=range(1, seqs_per_patient.max() + 2), color='#9b59b6', edgecolor='white', alpha=0.8, align='left')
ax.set_xlabel('Последовательностей у пациента')
ax.set_ylabel('Количество пациентов')
ax.set_title('Распределение последовательностей на пациента')
ax.axvline(seqs_per_patient.mean(), color='red', linestyle='--', label=f'mean={seqs_per_patient.mean():.1f}')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Столбчатая диаграмма: распределение классов по последовательностям
fig, ax = plt.subplots(figsize=(14, 4))

# Pie: class 0 vs class 1 bbox distribution
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))

ax2 = axes2[0]
ax2.pie([total_c0, total_c1], labels=['50-70% (class 0)', '>70% (class 1)'],
        autopct='%1.1f%%', colors=['#f39c12', '#c0392b'], startangle=90, textprops={'fontsize': 12})
ax2.set_title(f'Распределение bbox по классам\n(всего {total_c0 + total_c1} bbox)', fontsize=13)

# По сплитам: кадры со стенозом
ax2 = axes2[1]
split_data = df.groupby('split')['has_stenosis'].agg(['sum', 'count']).reindex(SPLITS)
split_data['no_stenosis'] = split_data['count'] - split_data['sum']
x = range(len(SPLITS))
width = 0.35
ax2.bar([i - width/2 for i in x], split_data['sum'], width, label='Со стенозом', color='#e74c3c', alpha=0.8)
ax2.bar([i + width/2 for i in x], split_data['no_stenosis'], width, label='Без стеноза', color='#2ecc71', alpha=0.8)
ax2.set_xticks(list(x))
ax2.set_xticklabels(SPLITS)
ax2.set_ylabel('Количество кадров')
ax2.set_title('Кадры по сплитам: стеноз vs норма')
ax2.legend()

plt.tight_layout()
plt.show()
plt.close(fig)  # close unused first fig

In [ ]:
# Столбчатая диаграмма: последовательности по пациентам (со стенозом / без)
fig, ax = plt.subplots(figsize=(16, 5))

patient_stats_sorted = patient_stats.sort_values('total_sequences', ascending=False)
x = range(len(patient_stats_sorted))
width = 0.4

ax.bar([i - width/2 for i in x], patient_stats_sorted['sequences_with_stenosis'], width, 
       label='Со стенозом', color='#e74c3c', alpha=0.8)
ax.bar([i + width/2 for i in x], patient_stats_sorted['sequences_without_stenosis'], width, 
       label='Без стеноза', color='#2ecc71', alpha=0.8)

ax.set_xticks(list(x))
ax.set_xticklabels(patient_stats_sorted['patient'], rotation=45, fontsize=9)
ax.set_xlabel('Пациент')
ax.set_ylabel('Количество последовательностей')
ax.set_title('Последовательности по пациентам: со стенозом vs без')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Распределение доли кадров со стенозом в последовательностях
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(seq_stats['stenosis_ratio'].values, bins=20, color='#e67e22', edgecolor='white', alpha=0.85)
ax.set_xlabel('Доля кадров со стенозом в последовательности')
ax.set_ylabel('Количество последовательностей')
ax.set_title('Распределение доли кадров со стенозом в последовательностях')
plt.tight_layout()
plt.show()

# Количество bbox-ов на кадр (среди кадров со стенозом)
stenosis_frames = df[df['has_stenosis']]
bbox_counts = stenosis_frames['num_bboxes'].value_counts().sort_index()
print(f"\nКоличество bbox-ов на кадр (среди кадров со стенозом):")
for n, count in bbox_counts.items():
    print(f"  {n} bbox: {count} кадров ({count/len(stenosis_frames)*100:.1f}%)")

## 8. Примеры кадров из нескольких последовательностей

Показаны кадры из случайных последовательностей (со стенозом и без). Красные bbox — класс 1 (>70%), оранжевые — класс 0 (50–70%).

In [ ]:
import random
random.seed(412)

seqs_with = seq_stats[seq_stats['has_any_stenosis']]['sequence_id'].tolist()
seqs_without = seq_stats[~seq_stats['has_any_stenosis']]['sequence_id'].tolist()

sample_with = random.sample(seqs_with, min(3, len(seqs_with)))
sample_without = random.sample(seqs_without, min(3, len(seqs_without)))
sample_seqs = sample_with + sample_without

N_FRAMES_PER_SEQ = 6
n_rows = len(sample_seqs)

fig, axes = plt.subplots(n_rows, N_FRAMES_PER_SEQ, figsize=(N_FRAMES_PER_SEQ * 3, n_rows * 3))

BBOX_COLORS = {0: '#f39c12', 1: '#e74c3c'}  # orange for class 0, red for class 1

for row_idx, seq_id in enumerate(sample_seqs):
    seq_df = df[df['sequence_id'] == seq_id].sort_values('frame')
    split = seq_df.iloc[0]['split']
    indices = np.linspace(0, len(seq_df) - 1, N_FRAMES_PER_SEQ, dtype=int)
    selected = seq_df.iloc[indices]
    
    has_st = seq_id in seqs_with
    label_text = "СТЕНОЗ" if has_st else "НОРМА"
    color = 'red' if has_st else 'green'
    
    for col_idx, (_, frame_row) in enumerate(selected.iterrows()):
        ax = axes[row_idx, col_idx]
        img_path = DATASET_DIR / split / "images" / frame_row['image_name']
        img = Image.open(img_path)
        ax.imshow(np.array(img), cmap='gray')
        
        if frame_row['has_stenosis']:
            lbl_name = frame_row['image_name'].replace('.png', '.txt')
            lbl_path = DATASET_DIR / split / "labels" / lbl_name
            content = lbl_path.read_text().strip()
            w_img, h_img = img.size
            for line in content.splitlines():
                parts = line.split()
                cls = int(parts[0])
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = (cx - bw / 2) * w_img
                y1 = (cy - bh / 2) * h_img
                rect = plt.Rectangle((x1, y1), bw * w_img, bh * h_img, 
                                     linewidth=2, edgecolor=BBOX_COLORS.get(cls, 'red'), facecolor='none')
                ax.add_patch(rect)
        
        ax.set_xticks([])
        ax.set_yticks([])
        if col_idx == 0:
            ax.set_ylabel(f"{seq_id}\n[{label_text}]", fontsize=10, color=color, fontweight='bold')
        ax.set_title(f"frame {frame_row['frame']}", fontsize=8)

plt.suptitle('Примеры последовательностей (верхние — со стенозом, нижние — без)\nОранжевый: 50-70%, Красный: >70%', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()